In [ ]:
import os
import gc
import json
import random
import re
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_addons as tfa
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# ─────────────────────────────────────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────────────────────────────────────

UNIT_MODE = "auto"  # "auto" | "seconds" | "minutes"
SEC_TO_MIN_THRESHOLD = 600.0  # if Δt95% > 600 (≈10 min in seconds), assume seconds and /60

# Your modified dataset folder
ROOT = "/path/to/prot_tools_unmod/"

# Your CSV files (modify/add as needed)
FILES = ["ptools_large.csv"]
OUTPUT_DIR = Path.cwd()

# Default model knobs (used if HP_SEARCH = False)
EPOCHS = 500
BATCH = 256
D_MODEL = 256
N_LAYERS = 16
N_HEADS = 8
D_FF = 1024
DROPOUT = 0.10
CONV_K = 9

HUBER_DELTA = 1.0
WEIGHT_DECAY = 1e-4
WARMUP_STEPS = 4000
MIN_LR = 1e-5
BASE_LR = 2e-3

# Hyperparameter search config
HP_SEARCH = True          # turn OFF if you just want the default config
EPOCHS_TUNE = 500          # max epochs per HP trial (with early stopping)

HP_CONFIGS = [
    {
        "name": "small_d192_l8",
        "D_MODEL": 192,
        "N_LAYERS": 8,
        "N_HEADS": 4,
        "D_FF": 768,
        "DROPOUT": 0.10,
        "BASE_LR": 2e-3,
    },
    {
        "name": "baseline_d256_l8",
        "D_MODEL": 256,
        "N_LAYERS": 8,
        "N_HEADS": 8,
        "D_FF": 1024,
        "DROPOUT": 0.10,
        "BASE_LR": 2e-3,
    },
    {
        "name": "deep_d256_l12",
        "D_MODEL": 256,
        "N_LAYERS": 12,
        "N_HEADS": 8,
        "D_FF": 1024,
        "DROPOUT": 0.15,
        "BASE_LR": 1.5e-3,
    },
    {
        "name": "wide_d320_l12",
        "D_MODEL": 320,
        "N_LAYERS": 12,
        "N_HEADS": 8,
        "D_FF": 1280,
        "DROPOUT": 0.15,
        "BASE_LR": 1.5e-3,
    },
]

# ─────────────────────────────────────────────────────────────────────────────
# GPU + Mixed precision
# ─────────────────────────────────────────────────────────────────────────────

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass

print("GPUs:", tf.config.list_physical_devices("GPU"))

from keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision:", mixed_precision.global_policy().name)


def set_seed(seed=42):
    tf.keras.utils.set_random_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_seed(42)


def free_model(model=None):
    """Release a model's graph/weights/optimizer state to avoid GPU OOM.

    The pipeline builds many models in one process (4 HP trials + 5 CV folds +
    1 final model). Without clearing the session between them, TensorFlow keeps
    every graph and its optimizer state resident, so GPU memory only grows and
    eventually runs out. Call this after you are done with each model.
    """
    if model is not None:
        del model
    gc.collect()
    tf.keras.backend.clear_session()

# ─────────────────────────────────────────────────────────────────────────────
# Tokenizer (extended alphabet for modified residues)
# ─────────────────────────────────────────────────────────────────────────────

BASE_AA = "ACDEFGHIKLMNPQRSTVWY"  # 20 canonical amino acids

# Pool of characters to use for modifications (anything not in BASE_AA)
MOD_CHAR_POOL = "".join(
    c
    for c in "1234567890abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!@#$%^&*()_+-=[]{}|;:'\",.<>/?`~"
    if c not in BASE_AA
)

PROSIT_CORE = BASE_AA
PROSIT_OX = BASE_AA + "o"  # for backward compatibility (if you ever have 'o' codes)
DP_ALPHABET = BASE_AA + MOD_CHAR_POOL  # big alphabet: AA + all possible mod chars
PAD = 0

def infer_alphabet(seqs):
    """
    Decide which alphabet to use.
    If we see any char not in BASE_AA, we assume modified tokens and use DP_ALPHABET.
    Otherwise, fallback to PROSIT_CORE / PROSIT_OX.
    """
    if any(any(c not in BASE_AA for c in s) for s in seqs):
        return DP_ALPHABET
    if any("o" in s for s in seqs):
        return PROSIT_OX
    return PROSIT_CORE

def build_tokenizer(alphabet):
    t = {c: i + 1 for i, c in enumerate(alphabet)}
    t["[CLS]"] = len(alphabet) + 1
    return t

def encode_sequence(seq, tok, max_len):
    ids = [tok["[CLS]"]] + [tok[c] for c in seq if c in tok]
    if len(ids) > max_len + 1:
        ids = ids[: max_len + 1]
    return ids + [PAD] * ((max_len + 1) - len(ids))

# ─────────────────────────────────────────────────────────────────────────────
# LR Schedule / Optimizer
# ─────────────────────────────────────────────────────────────────────────────

class WarmupCosine(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, warmup_steps, min_lr=1e-5, total_steps=200_000):
        super().__init__()
        self.base_lr = tf.cast(base_lr, tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
        self.min_lr = tf.cast(min_lr, tf.float32)
        self.total = tf.cast(total_steps, tf.float32)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)

        # Warmup phase
        warm = self.base_lr * tf.minimum(1.0, step / tf.maximum(1.0, self.warmup_steps))

        # Cosine decay phase
        progress = tf.clip_by_value(
            (step - self.warmup_steps) / tf.maximum(1.0, self.total - self.warmup_steps),
            0.0,
            1.0,
        )
        cosine = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (
            1.0 + tf.cos(np.pi * progress)
        )

        return tf.where(step < self.warmup_steps, warm, cosine)

# ─────────────────────────────────────────────────────────────────────────────
# Layers (unchanged model)
# ─────────────────────────────────────────────────────────────────────────────

class StripMask(tf.keras.layers.Layer):
    """Stops Keras mask propagation to silence harmless warnings."""
    def __init__(self):
        super().__init__()
        self.supports_masking = True

    def call(self, x):
        return x

    def compute_mask(self, inputs, mask=None):
        return None

class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, max_len_with_cls, d_model, dropout):
        super().__init__()
        self.supports_masking = True
        self.pos = tf.keras.layers.Embedding(
            max_len_with_cls, d_model, name="positional_embedding"
        )
        self.do = tf.keras.layers.Dropout(dropout)

    def call(self, token_emb, training=None):
        L = tf.shape(token_emb)[1]
        pos_ids = tf.range(L)
        x = token_emb + self.pos(pos_ids)[None, ...]
        return self.do(x, training=training)

    def compute_mask(self, inputs, mask=None):
        return mask

class ConvModule(tf.keras.layers.Layer):
    """Pointwise (GLU) → DepthwiseConv1D → BN → SiLU → Pointwise."""
    def __init__(self, d_model, kernel_size=9, dropout=0.1):
        super().__init__()
        self.pw1 = tf.keras.layers.Dense(2 * d_model)  # GLU gating
        self.dw = tf.keras.layers.DepthwiseConv1D(kernel_size, padding="same")
        self.bn = tf.keras.layers.BatchNormalization(momentum=0.9, epsilon=1e-5)
        self.act = tf.keras.layers.Activation(tf.nn.silu)
        self.pw2 = tf.keras.layers.Dense(d_model)
        self.do = tf.keras.layers.Dropout(dropout)

    def call(self, x, training=None):
        u, g = tf.split(self.pw1(x), 2, axis=-1)
        x = u * tf.keras.activations.sigmoid(g)
        x = self.dw(x)
        x = self.bn(x, training=training)
        x = self.act(x)
        x = self.pw2(x)
        return self.do(x, training=training)

class GEGLUFFN(tf.keras.layers.Layer):
    """GEGLU feed-forward: Dense(2*d_ff) -> GEGLU -> Dropout -> Dense(d_model)."""
    def __init__(self, d_ff, d_model, dropout):
        super().__init__()
        self.pre = tf.keras.layers.Dense(2 * d_ff)
        self.do = tf.keras.layers.Dropout(dropout)
        self.proj = tf.keras.layers.Dense(d_model)
        self.norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    def call(self, x, training=None):
        x = self.norm(x)
        h = self.pre(x)
        a, b = tf.split(h, 2, axis=-1)
        geglu = tf.keras.activations.gelu(a) * b
        geglu = self.do(geglu, training=training)
        return self.proj(geglu)

class EncoderBlock(tf.keras.layers.Layer):
    """
    Conformer-style macaron block:
      0.5*FFN1 -> MHSA -> ConvModule -> 0.5*FFN2
    """
    def __init__(self, d_model, n_heads, d_ff, dropout, conv_k):
        super().__init__()

        # Macaron FFNs (each has its own LayerNorm inside GEGLUFFN)
        self.ffn1 = GEGLUFFN(d_ff=d_ff, d_model=d_model, dropout=dropout)
        self.ffn2 = GEGLUFFN(d_ff=d_ff, d_model=d_model, dropout=dropout)

        # MHSA
        self.norm_attn = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads=n_heads,
            key_dim=d_model // n_heads,
            dropout=dropout,
        )
        self.do_attn = tf.keras.layers.Dropout(dropout)

        # Conv module
        self.conv_norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.conv = ConvModule(d_model, kernel_size=conv_k, dropout=dropout)

    def call(self, x, attn_mask, training=None):
        # Macaron FFN1 (pre-norm inside GEGLUFFN) with 0.5 residual
        x = x + 0.5 * self.ffn1(x, training=training)

        # MHSA (pre-norm)
        y = self.mha(
            self.norm_attn(x),
            self.norm_attn(x),
            attention_mask=attn_mask,
            training=training,
        )
        x = x + self.do_attn(y, training=training)

        # Conv module (pre-norm)
        y = self.conv(self.conv_norm(x), training=training)
        x = x + y

        # Macaron FFN2 (pre-norm inside GEGLUFFN) with 0.5 residual
        x = x + 0.5 * self.ffn2(x, training=training)

        return x

class TransformerEncoder(tf.keras.layers.Layer):
    def __init__(
        self,
        vocab_size,
        max_len_with_cls,
        d_model,
        d_ff,
        n_layers,
        n_heads,
        dropout,
        conv_k,
    ):
        super().__init__()
        self.embed = tf.keras.layers.Embedding(
            vocab_size,
            d_model,
            mask_zero=True,
            name="aa_embedding",
        )
        self.pos = PositionalEmbedding(max_len_with_cls, d_model, dropout)
        self.strip = StripMask()
        self.blocks = [
            EncoderBlock(d_model, n_heads, d_ff, dropout, conv_k)
            for _ in range(n_layers)
        ]
        self.final_norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    def call(self, token_ids, training=None):
        key_padding_mask = tf.not_equal(token_ids, 0)  # [B, L] bool

        x = self.embed(token_ids)  # [B, L, D]
        x = self.pos(x, training=training)
        x = self.strip(x)  # stop mask propagation to silence warnings

        attn_mask = tf.cast(key_padding_mask[:, None, :], tf.bool)  # [B, 1, L]

        for blk in self.blocks:
            x = blk(x, attn_mask, training=training)

        return self.final_norm(x), key_padding_mask

class MaskedMeanMax(tf.keras.layers.Layer):
    """Pools [B,L,D] with mask [B,L] → returns [mean, max], each [B,D]."""
    def call(self, inputs):
        x, mask = inputs
        mask = tf.cast(mask, x.dtype)[:, :, None]  # [B, L, 1]

        # Mean
        sum_x = tf.reduce_sum(x * mask, axis=1)  # [B, D]
        length = tf.reduce_sum(mask, axis=1)  # [B, 1] for safe broadcast
        length = tf.maximum(length, tf.constant(1.0, x.dtype))
        mean = sum_x / length  # [B, D]

        # Max (pad positions to very negative)
        very_neg = tf.cast(-1e4, x.dtype)
        x_masked = tf.where(tf.cast(mask, tf.bool), x, very_neg)
        maxp = tf.reduce_max(x_masked, axis=1)  # [B, D]

        return [mean, maxp]

class AttnPool(tf.keras.layers.Layer):
    """
    Single-head learned attention pooling over sequence.
    Input:  x [B, L, D], mask [B, L] (bool)
    Output: pooled [B, D]
    """
    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.query = self.add_weight(
            name="attn_query",
            shape=(1, 1, d_model),
            initializer="glorot_uniform",
            trainable=True,
        )

    def call(self, inputs):
        x, mask = inputs   # x: [B,L,D], mask: [B,L] bool
        B = tf.shape(x)[0]
        D = tf.shape(x)[-1]

        q = tf.cast(self.query, x.dtype)      # [1,1,D]
        q = tf.tile(q, [B, 1, 1])             # [B,1,D]

        # scores: [B,1,L]
        scale = tf.math.sqrt(tf.cast(D, x.dtype))
        scores = tf.matmul(q, x, transpose_b=True) / scale

        # mask: 1 for valid, 0 for pad
        mask_f = tf.cast(mask[:, None, :], x.dtype)   # [B,1,L]
        scores = scores + (1.0 - mask_f) * tf.cast(-1e4, x.dtype)

        attn = tf.nn.softmax(scores, axis=-1)         # [B,1,L]
        ctx = tf.matmul(attn, x)                      # [B,1,D]
        return ctx[:, 0, :]                           # [B,D]

# ─────────────────────────────────────────────────────────────────────────────
# Utilities (DATA LOADING + MODIFICATIONS + METRICS)
# ─────────────────────────────────────────────────────────────────────────────

def build_mod_type_mapping_from_df(df, mods_col):
    """
    Scan the Modifications column and assign a unique character to each
    (mod_name, aa) combination using MOD_CHAR_POOL.

    If we run out of characters (very unlikely), we map extra types to
    the last character in the pool.
    """
    mod_type_to_char = {}
    if mods_col is None or mods_col not in df.columns:
        return mod_type_to_char

    char_idx = 0

    for s in df[mods_col]:
        if not isinstance(s, str):
            continue
        s_clean = s.strip()
        if not s_clean or s_clean.lower() == "unmodified":
            continue

        # pattern: "1xOxidation [M12]; 1xAcetyl [K18]" or "2xCarbamidomethyl [C7; C11]"
        for n, name, inside in re.findall(r"(\d+)x([A-Za-z]+)\s*\[([^\]]+)\]", s_clean):
            name = name.strip()
            # inside may contain multiple positions: "M12", "C7; C11", "S6; S21"
            for aa, pos in re.findall(r"([A-Z]?)(\d+)", inside):
                aa = aa or None
                key = (name, aa)
                if key not in mod_type_to_char:
                    if char_idx < len(MOD_CHAR_POOL):
                        mod_type_to_char[key] = MOD_CHAR_POOL[char_idx]
                        char_idx += 1
                    else:
                        # If we ever run out, map to last char as catch-all
                        mod_type_to_char[key] = MOD_CHAR_POOL[-1]

    return mod_type_to_char


def encode_sequences_with_mods(df, seq_col, mods_col, mod_type_to_char):
    """
    For each peptide, replace modified residues by specific characters
    corresponding to their (mod_name, aa) type, preserving length.

    If there is no modification info or mapping, just clean and return
    the original sequences.
    """
    # No modification mapping → just clean up sequences
    if not mod_type_to_char or mods_col is None or mods_col not in df.columns:
        return (
            df[seq_col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", "", regex=True)
            .tolist()
        )

    new_seqs = []

    for seq, mods_str in zip(df[seq_col], df[mods_col]):
        seq = str(seq).strip().replace(" ", "")
        L = len(seq)
        # per-position modification char (None = unmodified)
        pos_mod_char = [None] * L

        if isinstance(mods_str, str):
            s_clean = mods_str.strip()
            if s_clean and s_clean.lower() != "unmodified":
                for n, name, inside in re.findall(
                    r"(\d+)x([A-Za-z]+)\s*\[([^\]]+)\]", s_clean
                ):
                    name = name.strip()
                    for aa, pos in re.findall(r"([A-Z]?)(\d+)", inside):
                        aa = aa or None
                        pos_idx = int(pos) - 1  # 1-based → 0-based
                        if pos_idx < 0 or pos_idx >= L:
                            # invalid position → ignore
                            continue
                        key = (name, aa)
                        ch = mod_type_to_char.get(key)
                        if ch is None:
                            # unseen combo (very rare) → ignore or map to last char
                            ch = MOD_CHAR_POOL[-1]
                        pos_mod_char[pos_idx] = ch

        # Build the encoded sequence: AA if unmodified, special char if modified
        chars = []
        for aa, mch in zip(seq, pos_mod_char):
            if mch is None:
                chars.append(aa)
            else:
                chars.append(mch)

        new_seqs.append("".join(chars))

    return new_seqs

def load_tsv(path):
    """
    Unified loader for CSV/TSV files with columns like:
      - Sequence, Modifications, RT

    Returns DataFrame with:
      - 'sequence': modification-aware string (AA + mod chars)
      - 'rt': numeric RT (as in file, seconds or minutes)
    Unit normalization is handled later by normalize_rt_units().
    """
    df = pd.read_csv(path, sep=None, engine="python")
    cols_lower = {c.lower(): c for c in df.columns}

    # Find sequence column
    seq_col = None
    for key in ["sequence", "peptide sequence", "peptide"]:
        for lc, orig in cols_lower.items():
            if key == lc or key in lc:
                seq_col = orig
                break
        if seq_col:
            break

    # Find RT column
    rt_col = None
    for key in ["rt", "retention time", "retention_time", "tr"]:
        for lc, orig in cols_lower.items():
            if key == lc or key in lc:
                rt_col = orig
                break
        if rt_col:
            break

    # Find Modifications column (optional)
    mods_col = None
    for lc, orig in cols_lower.items():
        if "mod" in lc:
            mods_col = orig
            break

    if seq_col is None or rt_col is None:
        raise ValueError(
            f"sequence/rt columns not found in {path}. Columns: {list(df.columns)}"
        )

    # Build mapping from modification type → char for this file
    mod_mapping = build_mod_type_mapping_from_df(df, mods_col)
    if mod_mapping:
        print(
            f"[Data] Detected {len(mod_mapping)} modification types in "
            f"{os.path.basename(path)}"
        )

    # Encode sequences with modification-aware alphabet
    seqs_encoded = encode_sequences_with_mods(df, seq_col, mods_col, mod_mapping)

    # RT numeric
    df_out = pd.DataFrame(
        {
            "sequence": seqs_encoded,
            "rt": pd.to_numeric(df[rt_col], errors="coerce"),
        }
    )

    # Clean / filter
    df_out["sequence"] = (
        df_out["sequence"]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", "", regex=True)
    )
    df_out = df_out.dropna(subset=["rt"])
    df_out = df_out[df_out["sequence"].str.len() > 0].reset_index(drop=True)
    df_out.attrs["mod_mapping"] = {
        f"{name}|{aa or ''}": ch
        for (name, aa), ch in mod_mapping.items()
    }

    return df_out

def make_ds(X, y=None, batch=BATCH, shuffle=False):
    ds = (
        tf.data.Dataset.from_tensor_slices((X, y))
        if y is not None
        else tf.data.Dataset.from_tensor_slices(X)
    )
    if shuffle:
        ds = ds.shuffle(len(X), seed=42)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

def pearson_r(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    if a.size < 2:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def p95_width(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.percentile(x, 97.5) - np.percentile(x, 2.5))

def residual_ci95(residuals):
    """95% confidence interval (2.5–97.5 percentile) of residuals in minutes."""
    r = np.asarray(residuals, dtype=np.float64)
    lo = float(np.percentile(r, 2.5))
    hi = float(np.percentile(r, 97.5))
    return lo, hi

def normalize_rt_units(y_raw, unit_mode="auto"):
    """
    Keep your original logic:
      - "minutes": use minutes
      - "seconds": /60
      - "auto": detect seconds by Δt95 threshold (SEC_TO_MIN_THRESHOLD)
    """
    y = np.asarray(y_raw, dtype=np.float64)
    dt95 = p95_width(y)

    if unit_mode == "minutes":
        return y, "minutes"
    if unit_mode == "seconds":
        return y / 60.0, "seconds→minutes(/60)"

    # auto mode: detect seconds vs minutes by spread
    if dt95 > SEC_TO_MIN_THRESHOLD:
        return y / 60.0, "seconds→minutes(/60)"
    else:
        return y, "minutes"

# ─────────────────────────────────────────────────────────────────────────────
# Model factory + Hyperparameter tuning (unchanged)
# ─────────────────────────────────────────────────────────────────────────────

def build_model_from_hp(hp, max_len_with_cls, vocab_size, steps_per_epoch, epochs):
    """
    Build a Conformer-lite model using a hyperparameter dict `hp`.
    """
    # Start each model on a clean session so graphs/optimizer state from
    # previously-built models don't accumulate in (GPU) memory.
    free_model()

    d_model = hp["D_MODEL"]
    n_layers = hp["N_LAYERS"]
    n_heads = hp["N_HEADS"]
    d_ff = hp["D_FF"]
    dropout = hp.get("DROPOUT", DROPOUT)
    base_lr = hp.get("BASE_LR", BASE_LR)

    inp = tf.keras.Input(
        shape=(max_len_with_cls,),
        dtype=tf.int32,
        name="tokens",
    )

    enc = TransformerEncoder(
        vocab_size=vocab_size,
        max_len_with_cls=max_len_with_cls,
        d_model=d_model,
        d_ff=d_ff,
        n_layers=n_layers,
        n_heads=n_heads,
        dropout=dropout,
        conv_k=CONV_K,
    )

    x, key_mask = enc(inp)  # x: [B, L, D], key_mask: [B, L]

    cls_tok = x[:, 0, :]  # [B, D]
    mean_p, max_p = MaskedMeanMax()([x, key_mask])  # [B, D], [B, D]

    attn_pool = AttnPool(d_model=d_model, name="attn_pool")
    attn_vec = attn_pool([x, key_mask])  # [B, D]

    # Combine four summaries: CLS, mean, max, attention-pooled
    feat = tf.keras.layers.Concatenate()(
        [cls_tok, mean_p, max_p, attn_vec]
    )  # [B, 4D]

    feat = tf.keras.layers.LayerNormalization(epsilon=1e-6)(feat)
    feat = tf.keras.layers.Dropout(dropout)(feat)
    feat = tf.keras.layers.Dense(
        d_model * 2,
        activation=tf.keras.activations.gelu,
    )(feat)
    feat = tf.keras.layers.Dropout(dropout)(feat)

    out = tf.keras.layers.Dense(
        1,
        activation="linear",
        dtype="float32",   # keep regression head in float32
    )(feat)

    model = tf.keras.Model(inp, out, name=f"rt_conformer_lite_{hp.get('name','hp')}")

    total_steps = max(1, steps_per_epoch * epochs)
    lr_sched = WarmupCosine(
        base_lr,
        warmup_steps=WARMUP_STEPS,
        min_lr=MIN_LR,
        total_steps=total_steps,
    )

    opt = tfa.optimizers.AdamW(
        learning_rate=lr_sched,
        weight_decay=WEIGHT_DECAY,
        epsilon=1e-8,
        global_clipnorm=1.0,
    )

    model.compile(
        optimizer=opt,
        loss=tf.keras.losses.Huber(delta=HUBER_DELTA),
    )

    return model

def tune_hyperparams(X_tr, y_tr, X_val, y_val, max_len_with_cls, vocab_size):
    """
    Simple manual hyperparameter search over HP_CONFIGS.
    Returns the best hp dict.
    """
    print(f"\n[HP SEARCH] Train size: {len(X_tr)}, Val size: {len(X_val)}")
    steps_per_epoch = max(1, len(X_tr) // BATCH)

    best_hp = None
    best_loss = np.inf

    for i, hp in enumerate(HP_CONFIGS):
        print(f"\n[HP {i+1}/{len(HP_CONFIGS)}] {hp['name']}")
        print("  config:", {k: v for k, v in hp.items() if k != "name"})

        model = build_model_from_hp(
            hp,
            max_len_with_cls=max_len_with_cls,
            vocab_size=vocab_size,
            steps_per_epoch=steps_per_epoch,
            epochs=EPOCHS_TUNE,
        )

        ds_tr = make_ds(X_tr, y_tr[:, None], batch=BATCH, shuffle=True)
        ds_val = make_ds(X_val, y_val[:, None], batch=BATCH)

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=100,
                restore_best_weights=True,
                verbose=1,
            )
        ]

        hist = model.fit(
            ds_tr,
            validation_data=ds_val,
            epochs=EPOCHS_TUNE,
            verbose=2,
            callbacks=callbacks,
        )

        val_loss = float(min(hist.history["val_loss"]))
        print(f"  best val_loss for {hp['name']}: {val_loss:.6f}")

        if val_loss < best_loss:
            best_loss = val_loss
            best_hp = hp

        # Done with this trial's model — free it before building the next one.
        free_model(model)
        del ds_tr, ds_val

    print(f"\n[HP SEARCH] Best config: {best_hp['name']} (val_loss={best_loss:.6f})")
    return best_hp

# ─────────────────────────────────────────────────────────────────────────────
# 5-fold cross validation (unchanged)
# ─────────────────────────────────────────────────────────────────────────────

def run_cross_validation(
    X,
    y_scaled,
    df,
    y_mean,
    y_std,
    best_hp,
    max_len_with_cls,
    vocab_size,
    file_stem,
):
    """
    5-fold cross-validation over the entire dataset using best_hp.
    Saves:
      - {stem}_cv_metrics.csv
      - {stem}_test_predictions_cv.csv
    """
    print("\n[CV] Running 5-fold cross validation...")
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    metrics_rows = []
    preds_rows = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        print(f"\n[CV] Fold {fold}/5")
        X_train_f = X[train_idx]
        X_val_f = X[val_idx]
        y_train_f = y_scaled[train_idx]
        y_val_f = y_scaled[val_idx]

        ds_train_f = make_ds(X_train_f, y_train_f[:, None], batch=BATCH, shuffle=True)
        ds_val_f = make_ds(X_val_f, y_val_f[:, None], batch=BATCH)

        steps_per_epoch = max(1, len(X_train_f) // BATCH)
        model = build_model_from_hp(
            best_hp,
            max_len_with_cls=max_len_with_cls,
            vocab_size=vocab_size,
            steps_per_epoch=steps_per_epoch,
            epochs=EPOCHS,
        )

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=100,
                restore_best_weights=True,
                verbose=1,
            )
        ]

        model.fit(
            ds_train_f,
            validation_data=ds_val_f,
            epochs=EPOCHS,
            verbose=2,
            callbacks=callbacks,
        )

        # Predictions on validation fold (treated as CV "test")
        ds_val_tokens = make_ds(X_val_f, batch=BATCH, shuffle=False)
        y_val_pred_scaled = model.predict(ds_val_tokens, verbose=0).reshape(-1)

        y_true = y_val_f * y_std + y_mean
        y_pred = y_val_pred_scaled * y_std + y_mean

        mse = mean_squared_error(y_true, y_pred)
        rmse = float(np.sqrt(mse))
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        R = pearson_r(y_true, y_pred)

        residuals = y_pred - y_true  # Pred - Exp
        ci_low, ci_high = residual_ci95(residuals)
        delta_t95 = ci_high - ci_low
        exp_t95 = p95_width(y_true)
        delta_tr95_pct = (
            float(100.0 * delta_t95 / exp_t95) if exp_t95 > 0 else np.nan
        )

        metrics_rows.append(
            dict(
                file=file_stem,
                fold=fold,
                n=len(val_idx),
                R=R,
                R2=r2,
                Dt95_min=delta_t95,
                Exp_Dt95_min=exp_t95,
                Dtr95_pct=delta_tr95_pct,
                MAE_min=mae,
                MSE_min2=mse,
                RMSE_min=rmse,
                CI_low_min=ci_low,
                CI_high_min=ci_high,
            )
        )

        # Save predictions for this fold
        for local_i, global_idx in enumerate(val_idx):
            preds_rows.append(
                dict(
                    file=file_stem,
                    fold=fold,
                    index=int(global_idx),
                    sequence=df.iloc[global_idx]["sequence"],
                    rt_true_min=float(y_true[local_i]),
                    rt_pred_min=float(y_pred[local_i]),
                )
            )

        # Done with this fold's model — free it before the next fold.
        free_model(model)
        del ds_train_f, ds_val_f, ds_val_tokens

    metrics_df = pd.DataFrame(metrics_rows).rename(
        columns={
            "R2": "R²",
            "Dt95_min": "Residual Δt₉₅% (min)",
            "Exp_Dt95_min": "Experimental Δt₉₅% (min)",
            "Dtr95_pct": "Residual Δt₉₅% / Experimental Δt₉₅% (%)",
            "MAE_min": "MAE (min)",
            "MSE_min2": "MSE (min²)",
            "RMSE_min": "RMSE (min)",
            "CI_low_min": "CI_low (min)",
            "CI_high_min": "CI_high (min)",
        }
    )
    cv_metrics_path = f"/path/to/prot_tools_unmod/{file_stem}_cv_metrics.csv"
    metrics_df.to_csv(cv_metrics_path, index=False)
    print("[CV] Saved metrics →", cv_metrics_path)

    preds_df = pd.DataFrame(preds_rows).sort_values(["fold", "index"])
    cv_preds_path = f"/path/to/prot_tools_unmod/{file_stem}_test_predictions_cv.csv"
    preds_df.to_csv(cv_preds_path, index=False)
    print("[CV] Saved predictions →", cv_preds_path)


# ─────────────────────────────────────────────────────────────────────────────
# Train & Evaluate one file
# ─────────────────────────────────────────────────────────────────────────────

def train_one_file(path):
    file_name = os.path.basename(path)
    file_stem = os.path.splitext(file_name)[0]

    print(f"\n=== {file_name} ===")
    df = load_tsv(path)          # <--- now loads & encodes modified peptides
    seqs = df["sequence"].tolist()

    # Units → minutes (auto-detect seconds vs minutes)
    y_minutes, detected = normalize_rt_units(df["rt"].values, unit_mode=UNIT_MODE)
    print(f"[Unit] Detected/used units for metrics: {detected}")

    # z-score target for training
    y_mean = float(y_minutes.mean())
    y_std_raw = y_minutes.std()
    y_std = float(y_std_raw if y_std_raw > 1e-6 else 1.0)
    y_scaled = ((y_minutes - y_mean) / y_std).astype(np.float32)

    alphabet = infer_alphabet(seqs)
    tok = build_tokenizer(alphabet)
    max_len = max(len(s) for s in seqs)
    max_len_with_cls = max_len + 1
    vocab_size = max(tok.values()) + 1

    X = np.stack([encode_sequence(s, tok, max_len) for s in seqs]).astype(np.int32)
    indices = np.arange(len(df))

    # Main train/test split (test is ONLY for final metrics; also used as val in final fit)
    idx_tr, idx_te, X_tr, X_te, y_tr, y_te = train_test_split(
        indices,
        X,
        y_scaled,
        test_size=0.20,
        random_state=42,
    )

    # Hyperparameter tuning uses extra val split from training data
    if HP_SEARCH:
        X_tr_sub, X_val, y_tr_sub, y_val = train_test_split(
            X_tr, y_tr, test_size=0.20, random_state=123
        )
        best_hp = tune_hyperparams(
            X_tr_sub,
            y_tr_sub,
            X_val,
            y_val,
            max_len_with_cls=max_len_with_cls,
            vocab_size=vocab_size,
        )
    else:
        best_hp = {
            "name": "manual_default",
            "D_MODEL": D_MODEL,
            "N_LAYERS": N_LAYERS,
            "N_HEADS": N_HEADS,
            "D_FF": D_FF,
            "DROPOUT": DROPOUT,
            "BASE_LR": BASE_LR,
        }
        print("\n[HP SEARCH] Disabled, using manual defaults:", best_hp)

    # 5-fold CV on full dataset (using best_hp)
    run_cross_validation(
        X,
        y_scaled,
        df,
        y_mean,
        y_std,
        best_hp,
        max_len_with_cls=max_len_with_cls,
        vocab_size=vocab_size,
        file_stem=file_stem,
    )

    # Final training on full training set with best hyperparameters
    steps_per_epoch = max(1, len(X_tr) // BATCH)
    model = build_model_from_hp(
        best_hp,
        max_len_with_cls=max_len_with_cls,
        vocab_size=vocab_size,
        steps_per_epoch=steps_per_epoch,
        epochs=EPOCHS,
    )

    ds_tr_full = make_ds(X_tr, y_tr[:, None], batch=BATCH, shuffle=True)
    ds_va = make_ds(X_te, y_te[:, None], batch=BATCH)   # used as val + later test

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=100,
            restore_best_weights=True,
            verbose=1,
        ),
    ]

    print("\n[TRAIN] Final model training with best hyperparameters...")
    history = model.fit(
        ds_tr_full,
        validation_data=ds_va,
        epochs=EPOCHS,
        verbose=2,
        callbacks=callbacks,
    )

    # ── Per-epoch training / validation loss CSV ──────────────────────────
    hist_df = pd.DataFrame(
        {
            "epoch": np.arange(1, len(history.history["loss"]) + 1),
            "loss": history.history["loss"],
            "val_loss": history.history["val_loss"],
        }
    )
    hist_path = f"/path/to/prot_tools_unmod/{file_stem}_train_history.csv"
    hist_df.to_csv(hist_path, index=False)
    print("[TRAIN] Saved train history →", hist_path)

    # Predict & invert scaling back to minutes (on final train + val/test)
    ds_tr_tokens_noshuf = make_ds(X_tr, batch=BATCH, shuffle=False)
    y_tr_pred_scaled = model.predict(ds_tr_tokens_noshuf, verbose=0).reshape(-1)
    y_tr_true = y_tr * y_std + y_mean
    y_tr_pred = y_tr_pred_scaled * y_std + y_mean

    ds_te_tokens = make_ds(X_te, batch=BATCH, shuffle=False)
    y_pred_scaled = model.predict(ds_te_tokens, verbose=0).reshape(-1)
    y_true = y_te * y_std + y_mean
    y_pred = y_pred_scaled * y_std + y_mean

    # ── Train / Validation prediction CSVs ────────────────────────────────
    train_pred_df = pd.DataFrame(
        {
            "index": idx_tr,
            "sequence": df.iloc[idx_tr]["sequence"].values,
            "rt_true_min": y_tr_true.astype(float),
            "rt_pred_min": y_tr_pred.astype(float),
        }
    )
    train_pred_path = f"/path/to/prot_tools_unmod/{file_stem}_train_predictions.csv"
    train_pred_df.to_csv(train_pred_path, index=False)
    print("[PRED] Saved train predictions →", train_pred_path)

    val_pred_df = pd.DataFrame(
        {
            "index": idx_te,
            "sequence": df.iloc[idx_te]["sequence"].values,
            "rt_true_min": y_true.astype(float),
            "rt_pred_min": y_pred.astype(float),
        }
    )
    val_pred_path = f"/path/to/prot_tools_unmod/{file_stem}_validation_predictions.csv"
    val_pred_df.to_csv(val_pred_path, index=False)
    print("[PRED] Saved validation/test predictions →", val_pred_path)

    # ── Metrics (in minutes) for final model ──────────────────────────────
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    R = pearson_r(y_true, y_pred)

    residuals = y_pred - y_true  # Pred - Exp
    ci_low, ci_high = residual_ci95(residuals)
    delta_t95 = ci_high - ci_low
    exp_t95 = p95_width(y_true)
    delta_tr95_pct = (
        float(100.0 * delta_t95 / exp_t95) if exp_t95 > 0 else np.nan
    )

    print(f"Samples (n): {len(df)} | MaxLen: {max_len}")
    print(f"Best HP    : {best_hp['name']}")
    print(f"R        : {R:.6f}")
    print(f"R²       : {r2:.6f}")
    print(f"MAE      : {mae:.6f} min")
    print(f"MSE      : {mse:.6f} min^2")
    print(f"RMSE     : {rmse:.6f} min")
    print(f"Δt95% residual window (Pred-Exp): {delta_t95:.6f} min")
    print(
        f"Δt95% / experimental Δt95%: {delta_tr95_pct:.3f} % "
        f"(experimental Δt95%: {exp_t95:.6f} min)"
    )
    print(
        f"95% CI (residuals, min): "
        f"[{ci_low:.6f}, {ci_high:.6f}]"
    )

    result = dict(
        file=file_name,
        best_hp=best_hp["name"],
        n=len(df),
        max_len=max_len,
        R=R,
        R2=r2,
        Dt95_min=delta_t95,
        Exp_Dt95_min=exp_t95,
        Dtr95_pct=delta_tr95_pct,
        MAE_min=mae,
        MSE_min2=mse,
        RMSE_min=rmse,
        CI_low_min=ci_low,
        CI_high_min=ci_high,
    )

    # Save final trained model + weights + preprocessing metadata for later evaluation.
    model_dir = f"/path/to/prot_tools_unmod/{file_stem}_final_model"
    weights_path = f"/path/to/prot_tools_unmod/{file_stem}_final_weights.weights.h5"
    metadata_path = f"/path/to/prot_tools_unmod/{file_stem}_final_model_metadata.json"

    model.save(model_dir, include_optimizer=False, save_format="tf")
    model.save_weights(weights_path)

    metadata = {
        "file": file_name,
        "best_hp": best_hp,
        "alphabet": alphabet,
        "tokenizer": tok,
        "pad_token_id": PAD,
        "cls_token_id": tok["[CLS]"],
        "max_len": max_len,
        "max_len_with_cls": max_len_with_cls,
        "vocab_size": vocab_size,
        "target_mean_min": y_mean,
        "target_std_min": y_std,
        "unit_mode": UNIT_MODE,
        "detected_units": detected,
        "mod_mapping": df.attrs.get("mod_mapping", {}),
    }
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    print("[SAVE] Saved final model ->", model_dir)
    print("[SAVE] Saved final weights ->", weights_path)
    print("[SAVE] Saved model metadata ->", metadata_path)

    # Done with the final model for this file — free it before the next file.
    free_model(model)

    return result


# ─────────────────────────────────────────────────────────────────────────────
# Run & Save
# ─────────────────────────────────────────────────────────────────────────────

results = []

for f in FILES:
    p = os.path.join(ROOT, f)
    if os.path.exists(p):
        results.append(train_one_file(p))
    else:
        print(f"Missing: {p}")

res_df = pd.DataFrame(results).rename(
    columns={
        "best_hp": "best_hp",
        "R2": "R²",
        "Dt95_min": "Residual Δt₉₅% (min)",
        "Exp_Dt95_min": "Experimental Δt₉₅% (min)",
        "Dtr95_pct": "Residual Δt₉₅% / Experimental Δt₉₅% (%)",
        "MAE_min": "MAE (min)",
        "MSE_min2": "MSE (min²)",
        "RMSE_min": "RMSE (min)",
        "CI_low_min": "CI_low (min)",
        "CI_high_min": "CI_high (min)",
    }
)

res_df = res_df[
    [
        "file",
        "best_hp",
        "n",
        "max_len",
        "R",
        "R²",
        "Residual Δt₉₅% (min)",
        "Experimental Δt₉₅% (min)",
        "Residual Δt₉₅% / Experimental Δt₉₅% (%)",
        "MAE (min)",
        "MSE (min²)",
        "RMSE (min)",
        "CI_low (min)",
        "CI_high (min)",
    ]
]

out_csv = "/path/to/prot_tools_unmod/final_metrics.csv"
res_df.to_csv(out_csv, index=False)

print("\nSaved metrics →", out_csv)
print("─" * 80)
print(res_df.to_string(index=False))
print("─" * 80)

2026-08-14 16:09:24.193388: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-14 16:09:24.193434: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-14 16:09:24.194721: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-14 16:09:24.201628: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-14 16:09:24.839165: W tensorflow/compiler/tf2

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3090 Ti, compute capability 8.6


2026-08-14 16:09:25.808992: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-08-14 16:09:25.848219: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-08-14 16:09:25.850492: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Mixed precision: mixed_float16

=== ptools_large.csv ===
[Unit] Detected/used units for metrics: minutes

[HP SEARCH] Train size: 640000, Val size: 160000

[HP 1/4] small_d192_l8
  config: {'D_MODEL': 192, 'N_LAYERS': 8, 'N_HEADS': 4, 'D_FF': 768, 'DROPOUT': 0.1, 'BASE_LR': 0.002}


2026-08-14 16:09:37.138349: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-08-14 16:09:37.140880: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-08-14 16:09:37.143075: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Epoch 1/500
2500/2500 - 333s - loss: 0.0490 - val_loss: 0.0344 - 333s/epoch - 133ms/step
Epoch 2/500
2500/2500 - 312s - loss: 0.0207 - val_loss: 0.0180 - 312s/epoch - 125ms/step
Epoch 3/500
2500/2500 - 312s - loss: 0.0180 - val_loss: 0.0158 - 312s/epoch - 125ms/step
Epoch 4/500
2500/2500 - 312s - loss: 0.0167 - val_loss: 0.0146 - 312s/epoch - 125ms/step
Epoch 5/500
2500/2500 - 312s - loss: 0.0158 - val_loss: 0.0165 - 312s/epoch - 125ms/step
Epoch 6/500
2500/2500 - 311s - loss: 0.0153 - val_loss: 0.0172 - 311s/epoch - 124ms/step
Epoch 7/500
2500/2500 - 311s - loss: 0.0147 - val_loss: 0.0159 - 311s/epoch - 124ms/step
Epoch 8/500
2500/2500 - 312s - loss: 0.0146 - val_loss: 0.0136 - 312s/epoch - 125ms/step
Epoch 9/500
2500/2500 - 311s - loss: 0.0142 - val_loss: 0.0147 - 311s/epoch - 125ms/step
Epoch 10/500
2500/2500 - 312s - loss: 0.0142 - val_loss: 0.0167 - 312s/epoch - 125ms/step
Epoch 11/500
2500/2500 - 311s - loss: 0.0141 - val_loss: 0.0122 - 311s/epoch - 125ms/step
Epoch 12/500
2500/2

INFO:tensorflow:Assets written to: /home/eemslab/rt_unmod/prot_tools_unmod/ptools_large_final_model/assets


[SAVE] Saved final model -> /home/eemslab/rt_unmod/prot_tools_unmod/ptools_large_final_model
[SAVE] Saved final weights -> /home/eemslab/rt_unmod/prot_tools_unmod/ptools_large_final_weights.weights.h5
[SAVE] Saved model metadata -> /home/eemslab/rt_unmod/prot_tools_unmod/ptools_large_final_model_metadata.json

Saved metrics → /home/eemslab/rt_unmod/prot_tools_unmod/final_metrics.csv
────────────────────────────────────────────────────────────────────────────────
            file          best_hp       n  max_len        R       R²  Residual Δt₉₅% (min)  Experimental Δt₉₅% (min)  Residual Δt₉₅% / Experimental Δt₉₅% (%)  MAE (min)  MSE (min²)  RMSE (min)  CI_low (min)  CI_high (min)
ptools_large.csv baseline_d256_l8 1000000       40 0.986827 0.973587              5.433676                 43.660052                                12.445418   1.038361    3.863453    1.965567     -2.808106        2.62557
────────────────────────────────────────────────────────────────────────────────


In [6]:
# Ablation study
# Run this cell after the main training cell. It reuses the layer/data utilities above.

import os
from pathlib import Path
import pandas as pd

ABLATION_RUN = True
ABLATION_EPOCHS = EPOCHS
ABLATION_PATIENCE = 100
ABLATION_BATCH = BATCH
ABLATION_SAVE_MODELS = False  # set True if you also want every ablation model saved
ABLATION_OUTPUT_XLSX = "/home/eemslab/lys_acetylation/B01_F0/ablation_study_results.xlsx"

ABLATION_VARIANTS = [
    {
        "name": "full_model",
        "description": "Full Conformer-lite model: modifications + MHSA + convolution + CLS/mean/max/attention pooling.",
        "use_modified_tokens": True,
        "use_mhsa": True,
        "use_conv": True,
        "use_ffn1": True,
        "use_ffn2": True,
        "pool_cls": True,
        "pool_mean": True,
        "pool_max": True,
        "pool_attn": True,
    },
    {
        "name": "no_modification_tokens",
        "description": "Uses raw peptide sequence only; ignores modification-specific residue tokens.",
        "use_modified_tokens": False,
        "use_mhsa": True,
        "use_conv": True,
        "use_ffn1": True,
        "use_ffn2": True,
        "pool_cls": True,
        "pool_mean": True,
        "pool_max": True,
        "pool_attn": True,
    },
    {
        "name": "no_conv_module",
        "description": "Removes the depthwise convolution module from each encoder block.",
        "use_modified_tokens": True,
        "use_mhsa": True,
        "use_conv": False,
        "use_ffn1": True,
        "use_ffn2": True,
        "pool_cls": True,
        "pool_mean": True,
        "pool_max": True,
        "pool_attn": True,
    },
    {
        "name": "no_self_attention",
        "description": "Removes multi-head self-attention; keeps convolution and FFN modules.",
        "use_modified_tokens": True,
        "use_mhsa": False,
        "use_conv": True,
        "use_ffn1": True,
        "use_ffn2": True,
        "pool_cls": True,
        "pool_mean": True,
        "pool_max": True,
        "pool_attn": True,
    },
    {
        "name": "no_attention_pool",
        "description": "Removes learned attention pooling; keeps CLS, masked mean, and masked max pooling.",
        "use_modified_tokens": True,
        "use_mhsa": True,
        "use_conv": True,
        "use_ffn1": True,
        "use_ffn2": True,
        "pool_cls": True,
        "pool_mean": True,
        "pool_max": True,
        "pool_attn": False,
    },
    {
        "name": "no_cls_pool",
        "description": "Removes CLS pooling; keeps masked mean, masked max, and learned attention pooling.",
        "use_modified_tokens": True,
        "use_mhsa": True,
        "use_conv": True,
        "use_ffn1": True,
        "use_ffn2": True,
        "pool_cls": False,
        "pool_mean": True,
        "pool_max": True,
        "pool_attn": True,
    },
    {
        "name": "mean_max_pool_only",
        "description": "Uses only masked mean and masked max pooling from the encoder output.",
        "use_modified_tokens": True,
        "use_mhsa": True,
        "use_conv": True,
        "use_ffn1": True,
        "use_ffn2": True,
        "pool_cls": False,
        "pool_mean": True,
        "pool_max": True,
        "pool_attn": False,
    },
    {
        "name": "cls_pool_only",
        "description": "Uses only the CLS token representation from the encoder output.",
        "use_modified_tokens": True,
        "use_mhsa": True,
        "use_conv": True,
        "use_ffn1": True,
        "use_ffn2": True,
        "pool_cls": True,
        "pool_mean": False,
        "pool_max": False,
        "pool_attn": False,
    },
]


def load_unmodified_tsv(path):
    """Load raw peptide sequences and RT, ignoring the Modifications column."""
    df = pd.read_csv(path, sep=None, engine="python")
    cols_lower = {c.lower(): c for c in df.columns}

    seq_col = None
    for key in ["sequence", "peptide sequence", "peptide"]:
        for lc, orig in cols_lower.items():
            if key == lc or key in lc:
                seq_col = orig
                break
        if seq_col:
            break

    rt_col = None
    for key in ["rt", "retention time", "retention_time", "tr"]:
        for lc, orig in cols_lower.items():
            if key == lc or key in lc:
                rt_col = orig
                break
        if rt_col:
            break

    if seq_col is None or rt_col is None:
        raise ValueError(f"sequence/rt columns not found in {path}. Columns: {list(df.columns)}")

    df_out = pd.DataFrame(
        {
            "sequence": df[seq_col].astype(str).str.strip().str.replace(r"\s+", "", regex=True),
            "rt": pd.to_numeric(df[rt_col], errors="coerce"),
        }
    )
    df_out = df_out.dropna(subset=["rt"])
    df_out = df_out[df_out["sequence"].str.len() > 0].reset_index(drop=True)
    df_out.attrs["mod_mapping"] = {}
    return df_out


def get_ablation_hp(file_stem):
    """Use the best HP from final_metrics.csv when available; otherwise fall back safely."""
    hp_by_name = {hp["name"]: hp for hp in HP_CONFIGS}
    final_metrics_path = "/home/eemslab/lys_acetylation/B01_F0/final_metrics.csv"

    #if final_metrics_path.exists():
    if os.path.exists(final_metrics_path):
        try:
            final_df = pd.read_csv(final_metrics_path)
            row = final_df[final_df["file"].astype(str).str.replace(r"\.[^.]+$", "", regex=True) == file_stem]
            if not row.empty:
                hp_name = str(row.iloc[0]["best_hp"])
                if hp_name in hp_by_name:
                    return dict(hp_by_name[hp_name])
        except Exception as exc:
            print(f"[ABLATION] Could not read final_metrics.csv for HP selection: {exc}")

    if HP_CONFIGS:
        return dict(HP_CONFIGS[0])

    return {
        "name": "manual_default",
        "D_MODEL": D_MODEL,
        "N_LAYERS": N_LAYERS,
        "N_HEADS": N_HEADS,
        "D_FF": D_FF,
        "DROPOUT": DROPOUT,
        "BASE_LR": BASE_LR,
    }


class AblationEncoderBlock(tf.keras.layers.Layer):
    """Conformer-style block with switches for ablation experiments."""
    def __init__(self, d_model, n_heads, d_ff, dropout, conv_k, variant):
        super().__init__()
        self.variant = variant
        self.use_mhsa = bool(variant.get("use_mhsa", True))
        self.use_conv = bool(variant.get("use_conv", True))
        self.use_ffn1 = bool(variant.get("use_ffn1", True))
        self.use_ffn2 = bool(variant.get("use_ffn2", True))

        if self.use_ffn1:
            self.ffn1 = GEGLUFFN(d_ff=d_ff, d_model=d_model, dropout=dropout)
        if self.use_ffn2:
            self.ffn2 = GEGLUFFN(d_ff=d_ff, d_model=d_model, dropout=dropout)

        if self.use_mhsa:
            self.norm_attn = tf.keras.layers.LayerNormalization(epsilon=1e-6)
            self.mha = tf.keras.layers.MultiHeadAttention(
                num_heads=n_heads,
                key_dim=d_model // n_heads,
                dropout=dropout,
            )
            self.do_attn = tf.keras.layers.Dropout(dropout)

        if self.use_conv:
            self.conv_norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)
            self.conv = ConvModule(d_model, kernel_size=conv_k, dropout=dropout)

    def call(self, x, attn_mask, training=None):
        if self.use_ffn1:
            x = x + 0.5 * self.ffn1(x, training=training)

        if self.use_mhsa:
            y = self.mha(
                self.norm_attn(x),
                self.norm_attn(x),
                attention_mask=attn_mask,
                training=training,
            )
            x = x + self.do_attn(y, training=training)

        if self.use_conv:
            x = x + self.conv(self.conv_norm(x), training=training)

        if self.use_ffn2:
            scale = 0.5 if self.use_ffn1 else 1.0
            x = x + scale * self.ffn2(x, training=training)

        return x


class AblationTransformerEncoder(tf.keras.layers.Layer):
    def __init__(self, vocab_size, max_len_with_cls, d_model, d_ff, n_layers, n_heads, dropout, conv_k, variant):
        super().__init__()
        self.embed = tf.keras.layers.Embedding(vocab_size, d_model, mask_zero=True, name="aa_embedding")
        self.pos = PositionalEmbedding(max_len_with_cls, d_model, dropout)
        self.strip = StripMask()
        self.blocks = [
            AblationEncoderBlock(d_model, n_heads, d_ff, dropout, conv_k, variant)
            for _ in range(n_layers)
        ]
        self.final_norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    def call(self, token_ids, training=None):
        key_padding_mask = tf.not_equal(token_ids, 0)
        x = self.embed(token_ids)
        x = self.pos(x, training=training)
        x = self.strip(x)
        attn_mask = tf.cast(key_padding_mask[:, None, :], tf.bool)

        for blk in self.blocks:
            x = blk(x, attn_mask, training=training)

        return self.final_norm(x), key_padding_mask


def build_ablation_model(variant, hp, max_len_with_cls, vocab_size, steps_per_epoch, epochs):
    free_model()

    d_model = hp["D_MODEL"]
    n_layers = hp["N_LAYERS"]
    n_heads = hp["N_HEADS"]
    d_ff = hp["D_FF"]
    dropout = hp.get("DROPOUT", DROPOUT)
    base_lr = hp.get("BASE_LR", BASE_LR)

    inp = tf.keras.Input(shape=(max_len_with_cls,), dtype=tf.int32, name="tokens")
    enc = AblationTransformerEncoder(
        vocab_size=vocab_size,
        max_len_with_cls=max_len_with_cls,
        d_model=d_model,
        d_ff=d_ff,
        n_layers=n_layers,
        n_heads=n_heads,
        dropout=dropout,
        conv_k=CONV_K,
        variant=variant,
    )
    x, key_mask = enc(inp)

    pool_vectors = []
    if variant.get("pool_cls", True):
        pool_vectors.append(x[:, 0, :])

    need_mean_max = variant.get("pool_mean", True) or variant.get("pool_max", True)
    if need_mean_max:
        mean_p, max_p = MaskedMeanMax()([x, key_mask])
        if variant.get("pool_mean", True):
            pool_vectors.append(mean_p)
        if variant.get("pool_max", True):
            pool_vectors.append(max_p)

    if variant.get("pool_attn", True):
        pool_vectors.append(AttnPool(d_model=d_model, name="attn_pool")([x, key_mask]))

    if not pool_vectors:
        raise ValueError(f"Variant {variant['name']} has no pooling output enabled.")

    feat = pool_vectors[0] if len(pool_vectors) == 1 else tf.keras.layers.Concatenate()(pool_vectors)
    feat = tf.keras.layers.LayerNormalization(epsilon=1e-6)(feat)
    feat = tf.keras.layers.Dropout(dropout)(feat)
    feat = tf.keras.layers.Dense(d_model * 2, activation=tf.keras.activations.gelu)(feat)
    feat = tf.keras.layers.Dropout(dropout)(feat)
    out = tf.keras.layers.Dense(1, activation="linear", dtype="float32")(feat)

    model = tf.keras.Model(inp, out, name=f"ablation_{variant['name']}")
    lr_sched = WarmupCosine(
        base_lr,
        warmup_steps=WARMUP_STEPS,
        min_lr=MIN_LR,
        total_steps=max(1, steps_per_epoch * epochs),
    )
    opt = tfa.optimizers.AdamW(
        learning_rate=lr_sched,
        weight_decay=WEIGHT_DECAY,
        epsilon=1e-8,
        global_clipnorm=1.0,
    )
    model.compile(optimizer=opt, loss=tf.keras.losses.Huber(delta=HUBER_DELTA))
    return model


def prepare_ablation_data(path, use_modified_tokens):
    df = load_tsv(path) if use_modified_tokens else load_unmodified_tsv(path)
    seqs = df["sequence"].tolist()
    y_minutes, detected = normalize_rt_units(df["rt"].values, unit_mode=UNIT_MODE)
    y_mean = float(y_minutes.mean())
    y_std_raw = y_minutes.std()
    y_std = float(y_std_raw if y_std_raw > 1e-6 else 1.0)
    y_scaled = ((y_minutes - y_mean) / y_std).astype(np.float32)

    alphabet = infer_alphabet(seqs)
    tok = build_tokenizer(alphabet)
    max_len = max(len(s) for s in seqs)
    max_len_with_cls = max_len + 1
    vocab_size = max(tok.values()) + 1
    X = np.stack([encode_sequence(s, tok, max_len) for s in seqs]).astype(np.int32)

    return {
        "df": df,
        "X": X,
        "y_scaled": y_scaled,
        "y_mean": y_mean,
        "y_std": y_std,
        "detected_units": detected,
        "alphabet": alphabet,
        "tokenizer": tok,
        "max_len": max_len,
        "max_len_with_cls": max_len_with_cls,
        "vocab_size": vocab_size,
    }


def ablation_metric_row(file_stem, variant_name, split, y_true, y_pred, best_hp_name, n, max_len, epochs_run, best_epoch, best_val_loss):
    residuals = y_pred - y_true
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    R = pearson_r(y_true, y_pred)
    q_low, q_high = residual_ci95(residuals)
    delta_t95 = q_high - q_low
    exp_t95 = p95_width(y_true)
    delta_t95_pct = float(100.0 * delta_t95 / exp_t95) if exp_t95 > 0 else np.nan

    return {
        "file": file_stem,
        "variant": variant_name,
        "split": split,
        "best_hp": best_hp_name,
        "n": n,
        "max_len": max_len,
        "epochs_run": epochs_run,
        "best_epoch": best_epoch,
        "best_val_loss_scaled": best_val_loss,
        "R": R,
        "R2": r2,
        "MAE_min": mae,
        "MSE_min2": mse,
        "RMSE_min": rmse,
        "Dt95_min": delta_t95,
        "Residual_Dt95_min": delta_t95,
        "Exp_Dt95_min": exp_t95,
        "residual_p2_5_min": q_low,
        "residual_p97_5_min": q_high,
        "Dtr95_abs_min": delta_t95,
        "Residual_Dt95_pct_of_exp": delta_t95_pct,
        "Dtr95_pct": delta_t95_pct,
        "residual_mean_min": float(np.mean(residuals)),
        "residual_median_min": float(np.median(residuals)),
        "residual_std_min": float(np.std(residuals)),
    }


def run_ablation_for_file(path):
    file_name = os.path.basename(path)
    file_stem = os.path.splitext(file_name)[0]
    best_hp = get_ablation_hp(file_stem)
    print(f"\n[ABLATION] {file_name} using HP: {best_hp['name']}")

    data_cache = {}
    metrics_rows = []
    history_rows = []
    prediction_rows = []
    config_rows = []
    data_rows = []

    for variant_i, variant in enumerate(ABLATION_VARIANTS, start=1):
        variant_name = variant["name"]
        use_mods = bool(variant.get("use_modified_tokens", True))
        print(f"\n[ABLATION {variant_i}/{len(ABLATION_VARIANTS)}] {variant_name}")
        print("  ", variant["description"])

        if use_mods not in data_cache:
            data_cache[use_mods] = prepare_ablation_data(path, use_mods)
        data = data_cache[use_mods]

        X = data["X"]
        y_scaled = data["y_scaled"]
        df = data["df"]
        indices = np.arange(len(df))

        idx_tr, idx_te, X_tr, X_te, y_tr, y_te = train_test_split(
            indices,
            X,
            y_scaled,
            test_size=0.20,
            random_state=42,
        )

        ds_tr = make_ds(X_tr, y_tr[:, None], batch=ABLATION_BATCH, shuffle=True)
        ds_te = make_ds(X_te, y_te[:, None], batch=ABLATION_BATCH)

        set_seed(42)
        model = build_ablation_model(
            variant,
            best_hp,
            max_len_with_cls=data["max_len_with_cls"],
            vocab_size=data["vocab_size"],
            steps_per_epoch=max(1, len(X_tr) // ABLATION_BATCH),
            epochs=ABLATION_EPOCHS,
        )

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=ABLATION_PATIENCE,
                restore_best_weights=True,
                verbose=1,
            )
        ]

        hist = model.fit(
            ds_tr,
            validation_data=ds_te,
            epochs=ABLATION_EPOCHS,
            verbose=2,
            callbacks=callbacks,
        )

        val_losses = hist.history.get("val_loss", [])
        best_epoch = int(np.argmin(val_losses) + 1) if val_losses else np.nan
        best_val_loss = float(np.min(val_losses)) if val_losses else np.nan
        epochs_run = len(hist.history.get("loss", []))

        for epoch_i in range(epochs_run):
            history_rows.append(
                {
                    "file": file_stem,
                    "variant": variant_name,
                    "epoch": epoch_i + 1,
                    "loss": hist.history["loss"][epoch_i],
                    "val_loss": hist.history["val_loss"][epoch_i],
                }
            )

        ds_tr_tokens = make_ds(X_tr, batch=ABLATION_BATCH, shuffle=False)
        ds_te_tokens = make_ds(X_te, batch=ABLATION_BATCH, shuffle=False)
        y_tr_pred_scaled = model.predict(ds_tr_tokens, verbose=0).reshape(-1)
        y_te_pred_scaled = model.predict(ds_te_tokens, verbose=0).reshape(-1)

        y_tr_true = y_tr * data["y_std"] + data["y_mean"]
        y_tr_pred = y_tr_pred_scaled * data["y_std"] + data["y_mean"]
        y_te_true = y_te * data["y_std"] + data["y_mean"]
        y_te_pred = y_te_pred_scaled * data["y_std"] + data["y_mean"]

        metrics_rows.append(
            ablation_metric_row(file_stem, variant_name, "train", y_tr_true, y_tr_pred, best_hp["name"], len(idx_tr), data["max_len"], epochs_run, best_epoch, best_val_loss)
        )
        metrics_rows.append(
            ablation_metric_row(file_stem, variant_name, "validation", y_te_true, y_te_pred, best_hp["name"], len(idx_te), data["max_len"], epochs_run, best_epoch, best_val_loss)
        )

        for split_name, split_idx, split_true, split_pred in [
            ("train", idx_tr, y_tr_true, y_tr_pred),
            ("validation", idx_te, y_te_true, y_te_pred),
        ]:
            for local_i, global_idx in enumerate(split_idx):
                prediction_rows.append(
                    {
                        "file": file_stem,
                        "variant": variant_name,
                        "split": split_name,
                        "index": int(global_idx),
                        "sequence": df.iloc[global_idx]["sequence"],
                        "rt_true_min": float(split_true[local_i]),
                        "rt_pred_min": float(split_pred[local_i]),
                        "residual_min": float(split_pred[local_i] - split_true[local_i]),
                    }
                )

        config_row = {"file": file_stem, "variant": variant_name, "description": variant["description"], **{k: v for k, v in variant.items() if k != "description"}}
        config_row.update({f"hp_{k}": v for k, v in best_hp.items()})
        config_rows.append(config_row)

        data_rows.append(
            {
                "file": file_stem,
                "variant": variant_name,
                "use_modified_tokens": use_mods,
                "n_total": len(df),
                "n_train": len(idx_tr),
                "n_validation": len(idx_te),
                "max_len": data["max_len"],
                "max_len_with_cls": data["max_len_with_cls"],
                "vocab_size": data["vocab_size"],
                "detected_units": data["detected_units"],
                "target_mean_min": data["y_mean"],
                "target_std_min": data["y_std"],
                "modification_types": len(df.attrs.get("mod_mapping", {})),
            }
        )

        if ABLATION_SAVE_MODELS:
            safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", variant_name)
            model_path = f"/home/eemslab/lys_acetylation/B01_F0/{file_stem}_ablation_{safe_name}_model"
            model.save(model_path, include_optimizer=False, save_format="tf")
            print("[ABLATION] Saved model ->", model_path)

        free_model(model)
        del ds_tr, ds_te, ds_tr_tokens, ds_te_tokens

    return metrics_rows, history_rows, prediction_rows, config_rows, data_rows


def run_ablation_study():
    if not ABLATION_RUN:
        print("[ABLATION] ABLATION_RUN is False; skipping.")
        return None

    all_metrics = []
    all_history = []
    all_predictions = []
    all_configs = []
    all_data = []

    for f in FILES:
        path = os.path.join(ROOT, f)
        if not os.path.exists(path):
            print(f"[ABLATION] Missing: {path}")
            continue

        metrics_rows, history_rows, prediction_rows, config_rows, data_rows = run_ablation_for_file(path)
        all_metrics.extend(metrics_rows)
        all_history.extend(history_rows)
        all_predictions.extend(prediction_rows)
        all_configs.extend(config_rows)
        all_data.extend(data_rows)

    metrics_df = pd.DataFrame(all_metrics)
    history_df = pd.DataFrame(all_history)
    predictions_df = pd.DataFrame(all_predictions)
    configs_df = pd.DataFrame(all_configs)
    data_df = pd.DataFrame(all_data)

    if not metrics_df.empty:
        val_df = metrics_df[metrics_df["split"] == "validation"].copy()
        val_df = val_df.sort_values(["file", "RMSE_min", "MAE_min", "Residual_Dt95_min"])
        best_df = val_df.groupby("file", as_index=False).head(1)
    else:
        val_df = pd.DataFrame()
        best_df = pd.DataFrame()

    with pd.ExcelWriter(ABLATION_OUTPUT_XLSX) as writer:
        metrics_df.to_excel(writer, sheet_name="metrics_all", index=False)
        val_df.to_excel(writer, sheet_name="validation_leaderboard", index=False)
        best_df.to_excel(writer, sheet_name="best_by_file", index=False)
        history_df.to_excel(writer, sheet_name="train_history", index=False)
        predictions_df.to_excel(writer, sheet_name="predictions", index=False)
        configs_df.to_excel(writer, sheet_name="variant_configs", index=False)
        data_df.to_excel(writer, sheet_name="data_summary", index=False)

    print("\n[ABLATION] Saved Excel workbook ->", ABLATION_OUTPUT_XLSX)
    if not val_df.empty:
        display_cols = ["file", "variant", "R", "R2", "MAE_min", "RMSE_min", "Residual_Dt95_min", "Residual_Dt95_pct_of_exp"]
        print("[ABLATION] Validation leaderboard:")
        print(val_df[display_cols].to_string(index=False))

    return {
        "metrics": metrics_df,
        "history": history_df,
        "predictions": predictions_df,
        "configs": configs_df,
        "data_summary": data_df,
        "validation_leaderboard": val_df,
        "best_by_file": best_df,
    }


ablation_results = run_ablation_study()


[ABLATION] B01_F0.csv using HP: baseline_d256_l8

[ABLATION 1/8] full_model
   Full Conformer-lite model: modifications + MHSA + convolution + CLS/mean/max/attention pooling.
[Data] Detected 5 modification types in B01_F0.csv
Epoch 1/500
41/41 - 24s - loss: 0.5752 - val_loss: 0.4592 - 24s/epoch - 586ms/step
Epoch 2/500
41/41 - 7s - loss: 0.3355 - val_loss: 0.1684 - 7s/epoch - 163ms/step
Epoch 3/500
41/41 - 7s - loss: 0.1755 - val_loss: 0.0818 - 7s/epoch - 163ms/step
Epoch 4/500
41/41 - 7s - loss: 0.0920 - val_loss: 0.0418 - 7s/epoch - 163ms/step
Epoch 5/500
41/41 - 7s - loss: 0.0614 - val_loss: 0.0420 - 7s/epoch - 162ms/step
Epoch 6/500
41/41 - 7s - loss: 0.0490 - val_loss: 0.0288 - 7s/epoch - 162ms/step
Epoch 7/500
41/41 - 7s - loss: 0.0412 - val_loss: 0.0384 - 7s/epoch - 161ms/step
Epoch 8/500
41/41 - 7s - loss: 0.0391 - val_loss: 0.0426 - 7s/epoch - 162ms/step
Epoch 9/500
41/41 - 7s - loss: 0.0407 - val_loss: 0.0246 - 7s/epoch - 162ms/step
Epoch 10/500
41/41 - 7s - loss: 0.0273 - v